# 1. 工具节点内审批

In [ ]:
from typing import Literal
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from dotenv import load_dotenv
from langgraph.types import interrupt, Command
from langchain_core.messages import ToolMessage

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools=tools)


# 定义状态
class State(MessagesState):
    pass


NEVER_ASK = {}


def need_approval(name: str, args: dict):
    return name not in NEVER_ASK


def tool_node(state: State) -> dict:
    last = state["messages"][-1]
    tool_map = {t.name: t for t in tools}

    # 1.收集所有审批决策
    decisions = {}
    for tc in last.tool_calls:
        if need_approval(tc['name'], tc['args']):
            decisions[tc['id']] = interrupt({"name": tc["name"], "args": tc["args"]})
        else:
            decisions[tc['id']] = True

    # 2. 决策齐了统一执行
    messages = []
    for tc in last.tool_calls:
        selected = tool_map.get(tc['name'])
        approved = decisions.get(tc['id'])
        if not selected:
            res = f"未知工具: {tc['name']}，请勿重试"
        elif not approved:
            res = f"用户拒绝执行 {tc['name']}，请勿重试，直接回答"
        else:
            res = tool_map[tc['name']].invoke(tc['args'])

        messages.append(ToolMessage(content=str(res), tool_call_id=tc['id']))

    return Command(goto="llm_node", update={"messages": messages})


def llm_node(state: State) -> Command[Literal["tool_node", END]]:
    ai_msg = model_with_tools.invoke(state["messages"])
    goto = "tool_node" if ai_msg.tool_calls else END
    return Command(goto=goto, update={"messages": [ai_msg]})


builder = StateGraph(state_schema=State)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}
graph = builder.compile(checkpointer)

res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气和科技相关的新闻")]}, config=config)
print(res)

In [ ]:
while res.get("__interrupt__"):
    info = res["__interrupt__"][0].value
    ans = input(f"允许 {info['name']}({info['args']})? (y/n) ").strip().lower() in ("y", "yes", "是", "1")
    res = graph.invoke(Command(resume=ans), config=config)  # ans 已经是 bool
print(res)

In [ ]:
print(list(graph.get_state_history(config=config)))

In [ ]:
from IPython.display import display

print(display(graph))